In [1]:
from datetime import datetime, date, timedelta
from tqdm import tqdm
from google.colab import drive

import numpy as np
import pandas as pd

In [2]:
"""
To make these data import code lines work:

1. Go to "Honors Thesis" Shared Drive.
2. Click the three vertical dots to the right of the "Exploratory Data Analysis" folder.
3. Click "Organize."
4. Click "Add shortcut."
5. Click the "Add" button to the right of "My Drive."

These steps create a shortcut in My Drive which points to the folder of data.

Now repeat for the last two DataFrames:

1. Go to "Honors Thesis" Shared Drive.
2. Click the three vertical dots to the right of the "Extracted Data Organized" folder.
3. Click "Organize."
4. Click "Add shortcut."
5. Click the "Add" button to the right of "My Drive."
"""

drive.mount('/content/drive')

eda1 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data1.csv")
eda2 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data2.csv")
eda3 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data3.csv")
eda4 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data4.csv")
eda5 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data5.csv")
eda6 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data6.csv")

ifact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/IL_update_fact.csv")
dfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/daily_fact.csv")

Mounted at /content/drive


/tmp/ipython-input-4203895347.py:23: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  eda1 = pd.read_csv("/content/drive/My Drive/Exploratory Data Analysis/eda_data1.csv")


In [3]:
def string_to_date(date_str): #convert strings to datetime
    return datetime.strptime(date_str, "%Y-%m-%d")

def date_to_string(date): #convert datetime to strings
    return date.strftime('%Y-%m-%d')

"""
rs_starts and rs_ends list the start and end of the regular seasons from 2017-18
to 2024-25. Each index from both lists is a single season.
"""
rs_starts = [
    datetime(2017, 10, 17),
    datetime(2018, 10, 16),
    datetime(2019, 10, 22),
    datetime(2020, 12, 22),
    datetime(2021, 10, 19),
    datetime(2022, 10, 18),
    datetime(2023, 10, 24),
    datetime(2024, 10, 22)
]

rs_ends = [
    datetime(2018, 4, 11),
    datetime(2019, 4, 10),
    datetime(2020, 4, 15),
    datetime(2021, 4, 16),
    datetime(2022, 4, 10),
    datetime(2023, 4, 9),
    datetime(2024, 4, 14),
    datetime(2025, 4, 13)
]

In [4]:
# identify significant injuries initially using the IL_update_fact table

# make ifact into lists for faster processing

pks = list(ifact['player_key'])
dates = list(ifact['date'])
details = list(ifact['details'])

season_ending_idx = [] # keeps indices of rows identified for season ending injuries

for i in range(len(details)): # loop through all descriptions from injury list updates

  if '(out for season)' in details[i]:
    season_ending_idx.append(i) # if the row states that the injury was season ending, mark that row

  else:
    continue

season_ending = [[pks[i], dates[i]] for i in season_ending_idx] # add the player key and date of injury to list of season ending injuries

# next identify the rows in the EDA data of are mark rows contain season ending injuries

# make eda1 into lists for faster processing

pks = list(eda1['player_key'])
dates = list(eda1['date'])

eda_season_ending = [] # for marking indices of EDA data for season ending injuries

print('Iterating through eda data for rows matching the season ending injuries from IL_update_fact...')

for i in tqdm(range(len(pks))): # for every row in EDA data...

  if [pks[i], dates[i]] in season_ending:
    eda_season_ending.append(i) # if the player_key and date exist in the season_ending list, mark row for season ending injury

  else:
    continue

print('Season ending injuries from IL_update_fact added...')
print('Total significant injuries so far: ', len(eda_season_ending))

# next, for every injury in eda data, change the injured? column value unless that row has been marked

updated_target = list(eda1['injured?']) # make target variable column into list

for i in range(len(updated_target)): # for every value in the target variable column

  if updated_target[i] == True: # if this is an injury...

    if i in eda_season_ending:
      continue # but it is significant, do not relabel this row

    else:
      updated_target[i] = False # otherwise, relabel this insignificant injury to False (not injured)

print('Updated target variable relabeled')

# finally use the updated values to replace the target variable on all six tables

print('Concatenating new columns...')

target_variable = pd.DataFrame({'injured?':updated_target})

eda1.drop('injured?', axis=1, inplace=True)
eda2.drop('injured?', axis=1, inplace=True)
eda3.drop('injured?', axis=1, inplace=True)
eda4.drop('injured?', axis=1, inplace=True)
eda5.drop('injured?', axis=1, inplace=True)
eda6.drop('injured?', axis=1, inplace=True)

eda1 = pd.concat([eda1, target_variable], axis=1)
eda2 = pd.concat([eda2, target_variable], axis=1)
eda3 = pd.concat([eda3, target_variable], axis=1)
eda4 = pd.concat([eda4, target_variable], axis=1)
eda5 = pd.concat([eda5, target_variable], axis=1)
eda6 = pd.concat([eda6, target_variable], axis=1)

Iterating through eda data for rows matching the season ending injuries from IL_update_fact...


100%|██████████| 537910/537910 [00:19<00:00, 28271.25it/s]


Season ending injuries from IL_update_fact added...
Total significant injuries so far:  433
Updated target variable relabeled
Concatenating new columns...


In [5]:
eda1.to_csv('season_ending_eda_data1.csv', index=False)
eda2.to_csv('season_ending_eda_data2.csv', index=False)
eda3.to_csv('season_ending_eda_data3.csv', index=False)
eda4.to_csv('season_ending_eda_data4.csv', index=False)
eda5.to_csv('season_ending_eda_data5.csv', index=False)
eda6.to_csv('season_ending_eda_data6.csv', index=False)

In [6]:
eda1['injured?'].value_counts()

,count
injured?,
False,537478
True,432
